In [1]:
import os
import json
from models import LLMModel

def read_prompt(name):
    with open(os.path.join('prompts', name), 'r') as f:
        prompt = f.read()
    return prompt 


In [2]:
def write_heuristics(dict_data, experiment = None):
    # Для словаря эвристик
    folder_path = os.path.join('Heuristics', experiment)
    os.makedirs(folder_path, exist_ok=True)
    for name, code in dict_data.items():
        with open(os.path.join(folder_path, name), "w", encoding="utf-8") as file:
            file.write(code)
def write_heuristic(code, name, experiment = None):
    # Для эвристики
    folder_path = os.path.join('Heuristics', experiment)
    os.makedirs(folder_path, exist_ok=True)
    with open(os.path.join(folder_path, name), "w", encoding="utf-8") as file:
        file.write(code)
    print(f'{name} save to {folder_path}')
def write_steps(dict_data, experiment = None):
    # Для шагов 
    folder_path = os.path.join('Heuristics', experiment, 'Steps')
    os.makedirs(folder_path, exist_ok=True)
    with open(os.path.join(folder_path, 'description.json'), 'w', encoding='utf-8') as f:
        json.dump(dict_data, f, ensure_ascii=False, indent=4)

In [ ]:
import json
import re

def extract_code(code_text, tag = 'python'): 
    # Extract code from code block
    pattern = fr'```(?:{tag})?\s*\n(.*?)```'
    code_blocks = re.findall(pattern, code_text, re.DOTALL)
    if code_blocks: 
        return code_blocks[0].strip() 
    else: 
        return None
    

# def extract_code(code_text, ): 
#     # Extract code from code block
#     pattern = r'```(?:python)?\s*\n(.*?)```'
#     code_blocks = re.findall(pattern, code_text, re.DOTALL)
#     if code_blocks: 
#         return code_blocks[0].strip() 
#     else: 
#         return None

### Method
def SGE(model_name,
        problem = 'RCPSP',
        EXPERIMENT = None):
    '''
    args:
    model (obj): Интерфейс к LLM-модели.
    problem (str): Тип RCPSP проблемы, RCPSP : default - классическая проблема.
    '''
    #model = create_model(max_tokens=None, model_name = model_name)
    model = LLMModel(model_name, max_tokens=15_000)
    z_exp, z_decomposition, z_integrate = read_prompt('1. exploration'), read_prompt('2. decomposition'), read_prompt('3. integrate')
    folder_path = os.path.join('Heuristics', EXPERIMENT)
    os.makedirs(folder_path, exist_ok=True)
    #  Step 1 - Generate heuristics name
    prompt1 = z_exp.format(problem = problem)
    # methods_text = model.send_message(prompt1) 
    # FOR CLAUDE
    methods_text = extract_code(model.send_message(prompt1) , tag = 'json')
    print(methods_text)
    try:
        methods = json.loads(methods_text)
        assert isinstance(methods, list) and all(isinstance(m, str) for m in methods)
    except Exception as  e:
        print('Error parsing methods: {e}')
    Q_N = methods[:10]
    print("Methods:", Q_N)
    # # Step 2 - Decomposition 
    Q_S = dict.fromkeys(Q_N)
    steps_raw = dict.fromkeys(Q_N)
    for k in range(len(Q_N)):
        Q_n = Q_N[k]
        prompt2 = z_decomposition.format(method=Q_n, problem = problem)
        #steps_text = model.send_message(prompt2)
          # FOR CLAUDE
        steps_text = extract_code(model.send_message(prompt2), tag='json')
        
        steps_raw[Q_n] = steps_text
        try:
            steps = json.loads(steps_text)            
            assert isinstance(steps, list) and all(isinstance(s, str) for s in steps)
            steps_str = '\n'.join(steps)
            Q_S[Q_n] = steps_str 
        except Exception as e:
            print(f"Error parsing steps for method {Q_n}: {e}")
            continue
    print("Steps:", Q_S)
    write_steps(Q_S, EXPERIMENT)
    #
    # with open('Heuristics/deepseek_chat_free/Steps/description.json', "r", encoding="utf-8") as f:
    #     Q_S = json.load(f)

    #Q_S = {k: Q_S[k] for k in ['Minimum Slack Priority Rule (MSP)',
    #  'Greatest Resource Demand Priority Rule (GRD)', 'Most Successors Priority Rule (MSP)']}
    #
    Q_A = dict.fromkeys(Q_S)
    # Step 3 - Write code 
    for method, steps_str in Q_S.items():
        prompt3 = z_integrate.format(method=method, steps=steps_str)
        python_txt = extract_code(model.send_message(prompt3, mode='coder'))
        Q_A[method] = python_txt
        write_heuristic(python_txt, method, EXPERIMENT)
    # Code, Input to model, Steps
    return Q_A

In [5]:
# EXPERIMENT = 'deepseek_reasoner'

# A,  S = SGE(model_name='deepseek-reasoner', EXPERIMENT=EXPERIMENT) 

# # 18 m -> 12 alg
# # 20 m -> 8 alg (reasoner)

In [6]:
models=  [
    "claude-sonnet-4.6", "claude-haiku-4.5"]


 #   "iairlab/qwen2.5-72b",

# "iairlab/qwen3-32b",


    # "iairlab/qwen3.5-35b-a3b",
    # "iairlab/qwen2.5-72b",
    # "iairlab/qwen3-32b-reasoning-cache",
    # "iairlab/qwen3-32b",
    # "iairlab/gpt-oss-120b",
    # "iairlab/qwen2.5-vl-72b",
    # "iairlab/qwen2.5-vl-7b",
    # "iairlab/e5-mistral-7b-instruct",
    # "iairlab/generation-model",

# model.split('/')[1].replace('-','_')

for model in models:
    EXPERIMENT = model.replace('-','_')
    print(f'Start for {EXPERIMENT}')
    #model = LLMModel(model)
    try:
       _, _ = SGE(model, EXPERIMENT=EXPERIMENT)
    except TimeoutError as e:
        print(f'Timeout in {EXPERIMENT}: {e}')
    except Exception as e:
        print(f'Failed for {EXPERIMENT}: {type(e).__name__}: {e}')

Start for claude_sonnet_4.6
[
  "Earliest Start Time Priority Rule (EST)",
  "Latest Start Time Priority Rule (LST)",
  "Earliest Finish Time Priority Rule (EFT)",
  "Latest Finish Time Priority Rule (LFT)",
  "Minimum Slack Priority Rule (MSLK)",
  "Most Total Successors Priority Rule (MTS)",
  "Most Immediate Successors Priority Rule (MIS)",
  "Greatest Resource Demand Priority Rule (GRD)",
  "Greatest Cumulative Resource Demand Priority Rule (GCRD)",
  "Most Resources Priority Rule (MR)",
  "Shortest Processing Time Priority Rule (SPT)",
  "Longest Processing Time Priority Rule (LPT)",
  "Resource Over Demand Priority Rule (ROD)",
  "Critical Path Priority Rule (CPM)",
  "Weighted Resource Utilization Priority Rule (WRU)",
  "Maximum Resource Requirement Priority Rule (MAXR)",
  "Minimum Resource Requirement Priority Rule (MINR)",
  "Activity Priority by Total Float (TF)",
  "Parallel Schedule Generation Scheme with Random Priority Rule (PSGS-RND)",
  "Parallel Schedule Generation S

# Valid heuristics
`Требования к эвристике:`

    - Временная сложность (timeout)
    - Ограничения по ресурсам (demand_min и т. д.)
    - Ограничения графа активностей (t_i > t_j, где t_i -- начало назначаемой работы должно быть позже времени окончания работы необходимых предшествующих работ j)

In [7]:
import tempfile

def my_interpter_solver(method, code, data):
    
    communication_coefficient = lambda n, m: 1.0 / (6 * m ** 2) * (-2 * n ** 3 + 3 * n ** 2 + (6 * m ** 2 - 1) * n) if m != 0 else 0.0
    with tempfile.NamedTemporaryFile(delete=False, suffix='.py') as temp_module:
        temp_module_name = os.path.basename(temp_module.name).split('.')[0]
        temp_module.write(code.encode('utf-8'))
        temp_module_path = temp_module.name
        jobs, resources_borders = data['jobs'], data['resources_detailed']
        try:
            # Import the module
            import importlib.util
            spec = importlib.util.spec_from_file_location(temp_module_name, temp_module_path)
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)
            # Check if 'rcpsp_solver' function exists
            if hasattr(module, 'rcpsp_solver'):
                schedule, order, resource_usage, d, makespan = module.rcpsp_solver(jobs, resources_borders, communication_coefficient)
                if not schedule or not order:
                    print(f"rcpsp_solver did not return a valid solution for method {method}")
            else:
                print(f"No rcpsp_solver function found in the code for method {method}")

            print('Solution:', makespan)

            return schedule, order, resource_usage, d, makespan

        except Exception as e:
                print(f"Error executing code for method {method}: {e}")
        finally:
                # Clean up the temporary file
                os.remove(temp_module_path)
                


In [12]:
import os
import json
from scripts.valid import *
import pandas as pd
from sampo.schemas.graph import WorkGraph
from sampo_api import *
from scripts.wg_converter import *




def test(heuristic, code, data):
    try:
        schedule, order, resource_usage, job_usage, makespan = my_interpter_solver(heuristic, code, data)
        return  (check_feasibility(schedule, job_usage, resource_usage,data), makespan)
    except Exception:
        return (False, -1)


def test1():
    # Проверка на простой кейс
    with open('testdata.json', 'r', encoding='utf-8') as file:
        data = json.load(file)['test0']
    return data
def test2():
    # Проверка на рабочий кейс
    wg = WorkGraph.loadf(f'wgs/small_synth', f'wg_9')
    contractors = contractor(N = 5)
    conv = WorkGraphConverter()
    data = conv.convert(wg, contractors)['rcpsp_data']
    return data
def test3():
    # Cycle for several problems
    pass


def test_heuristics(folder_path, heuristic):
    file = os.path.join(folder_path, heuristic)
    cases = dict.fromkeys(['name','test1', 'test2'])
    with open(file, "r", encoding="utf-8") as f:
        code = f.read()
    cases['name'] = heuristic
    # Test 1, assert makespan ~ 8
    data = test1()
    cases['test1'] = test(heuristic, code, data)
    # Test 2, wg_9.json, HEFT - 81, LFT - 58, GA - 59
    data = test2()
    cases['test2'] = test(heuristic, code, data)
    cases['test2_UPgap'] = round(100 * (cases['test2'][1] / 58 - 1))
    cases['test2_LOWgap'] = max( round(100 * (cases['test2'][1] / 81 - 1)), 0 )
    return cases
        
def test_prompt(EXPERIMENT):
    folder_path = os.path.join('Heuristics', EXPERIMENT)
    res = []
    for heuristic in os.listdir(folder_path)[1:]:
        if heuristic not in ('Steps', 'original_heuristics.json',) :
            print('_' * 12, heuristic, '_' * 12)
           # check, makespan = test_heuristics(os.path.join(folder_path, heuristic))
            res.append( test_heuristics(folder_path, heuristic) )
    return res


pd.DataFrame(test_prompt('claude_haiku_4.5'))

____________ Latest Start Time Priority Rule (LST) ____________
Solution: 8
Solution: 95
____________ Shortest Processing Time Priority Rule (SPT) ____________
Solution: 8
Solution: 37
____________ Greatest Resource Demand Priority Rule (GRD) ____________
Solution: 8
Solution: 70
____________ Earliest Finish Time Priority Rule (EFT) ____________
Error executing code for method Earliest Finish Time Priority Rule (EFT): free variable 'compute_duration' referenced before assignment in enclosing scope
Error executing code for method Earliest Finish Time Priority Rule (EFT): free variable 'compute_duration' referenced before assignment in enclosing scope
____________ Most Predecessors Priority Rule (MP) ____________
Solution: 8
Error executing code for method Most Predecessors Priority Rule (MP): Deadlock detected
____________ Most Successors Priority Rule (MS2) ____________
Solution: 8
Solution: 73
____________ Earliest Start Time Priority Rule (EST) ____________
Error executing code for m

,name,test1,test2,test2_UPgap,test2_LOWgap
0,Latest Start Time Priority Rule (LST),"(True, 8)","(True, 95)",64,17
1,Shortest Processing Time Priority Rule (SPT),"(True, 8)","(True, 37)",-36,0
2,Greatest Resource Demand Priority Rule (GRD),"(True, 8)","(True, 70)",21,0
3,Earliest Finish Time Priority Rule (EFT),"(False, -1)","(False, -1)",-102,0
4,Most Predecessors Priority Rule (MP),"(True, 8)","(False, -1)",-102,0
5,Most Successors Priority Rule (MS2),"(True, 8)","(True, 73)",26,0
6,Earliest Start Time Priority Rule (EST),"(False, -1)","(False, -1)",-102,0
7,Minimum Slack Priority Rule (MS1),"(True, 8)","(True, 118)",103,46
8,Latest Finish Time Priority Rule (LFT),"(True, 8)","(True, 124)",114,53


In [10]:
pd.DataFrame(test_prompt('gpt_5.4_nano'))

____________ Longest Processing Time Priority Rule (LPT) ____________
Solution: 8
Solution: 131
____________ Minimum Slack Priority Rule (MSP) ____________
Solution: 8
Solution: 130
____________ Parallel Schedule Generation Scheme (PGS) ____________
Solution: 8
Solution: 92
____________ List Scheduling with Priority Rules (LSP) ____________
Solution: 8
Solution: 100
____________ Serial Schedule Generation Scheme (SSGS) ____________
Solution: 8
Solution: 671
____________ Earliest Finish Time Priority Rule (EFT) ____________
Solution: 8
Solution: 135
____________ Most Successors Priority Rule (MSU) ____________
Solution: 8
Solution: 117
____________ Earliest Start Time Priority Rule (EST) ____________
Solution: 8
Solution: 92
____________ Highest Level Priority Rule (HL) ____________
Solution: 8
Solution: 130


,name,test1,test2,test2_UPgap,test2_LOWgap
0,Longest Processing Time Priority Rule (LPT),"(True, 8)","(True, 131)",126,62
1,Minimum Slack Priority Rule (MSP),"(True, 8)","(True, 130)",124,60
2,Parallel Schedule Generation Scheme (PGS),"(True, 8)","(True, 92)",59,14
3,List Scheduling with Priority Rules (LSP),"(True, 8)","(True, 100)",72,23
4,Serial Schedule Generation Scheme (SSGS),"(True, 8)","(True, 671)",1057,728
5,Earliest Finish Time Priority Rule (EFT),"(True, 8)","(True, 135)",133,67
6,Most Successors Priority Rule (MSU),"(True, 8)","(False, 117)",102,44
7,Earliest Start Time Priority Rule (EST),"(True, 8)","(False, 92)",59,14
8,Highest Level Priority Rule (HL),"(False, 8)","(False, 130)",124,60


In [11]:
pd.DataFrame(test_prompt('gemini_3.1_flash_lite_preview'))

____________ Longest Processing Time Priority Rule (LPT) ____________
Error executing code for method Longest Processing Time Priority Rule (LPT): Deadlock detected
Error executing code for method Longest Processing Time Priority Rule (LPT): Deadlock detected
____________ Latest Start Time Priority Rule (LST) ____________
Error executing code for method Latest Start Time Priority Rule (LST): '<' not supported between instances of 'dict' and 'dict'
Error executing code for method Latest Start Time Priority Rule (LST): '<' not supported between instances of 'dict' and 'dict'
____________ Shortest Processing Time Priority Rule (SPT) ____________
Solution: 8
Solution: 74
____________ Greatest Resource Demand Priority Rule (GRD) ____________


KeyboardInterrupt: 

In [ ]:
# with open('Heuristics/deepseek_chat/Earliest Start Time Priority Rule (EST)', "r", encoding="utf-8") as f:
#         code = f.read()
# data = test2()
# schedule, order, resource_usage, job_usage, makespan = my_interpter_solver('x', code, data)

# print( check_resource_feasibility(resource_usage, data['resources_detailed']) )

# print( find_capacity_violations(data['jobs'], data['resources_detailed'], schedule, job_usage) )



In [ ]:
# DeepSeek 

# def create_model(max_tokens=None, model_name = 'deepseek-chat'):
#     api_key = os.getenv("DEEPSEEK_API_KEY")
#     if not max_tokens:
#         return DeepSeekSession(api_key, '', 
#                                'You are an expert in resource constrained project scheduling problem algorithms. ' \
#                                'Write and run code to answer questions if needed.', model_name=model_name)
#     return DeepSeekSession(api_key, '', 
#                            '''
# You are an expert in resource constrained project scheduling problem algorithms.' \
# Write clean, executable, production-ready Python code. \
#                            ''',
#                             max_tokens=max_tokens,
#                             model_name=model_name)

In [ ]:
16 * 10_000 + 5_000